# IdeaFoundry - Capstone
Team Members: Rawan Alahmadi, Mayar Alhindi, Ghaidaa Alshareef

Program: Building Agentic AI Systems, SDAIA Academy

Trainer: Mohammed Albeladi

Cohort: 23 Aug – 27 Aug 2026

Track: Track A — Supervisor + Workers

Workflow Pattern: Orchestrator-Worker

## Executive summary

**IdeaFoundry** is an end-to-end multi-agent system that evaluates early-stage business
proposals and returns a defensible verdict.

**Intake and delegation.** A founder submits a business idea. The Supervisor reads it into a
structured brief, then delegates by emitting explicit `transfer_to_*` handoff tool calls. The
calls themselves *are* the routing decision, and we print them at Gate 1 as direct evidence
that the LLM - not a keyword rule - chose the analysts.

**Gate 0 - clarification (conditional).** An idea too vague to analyse pauses for answers
instead of being rejected, then is re-read and re-routed. This is the only route that produces
no report, which is what proves the Supervisor is deciding rather than dispatching.

**Gate 1 - human-in-the-loop.** The workflow pauses before any analyst runs, so the founder can
correct the brief or drop workers. A correction re-derives the brief, so the resume genuinely
changes the outcome.

**Parallel analyst execution.** The selected workers run concurrently, each holding only its own
tools and blind to the others. They ground their work in a Saudi business-setup corpus
(Monsha'at where reachable, other authoritative sources via live search otherwise) plus live
web search for competitors, labelling every finding `evidence` or `assumption`. A
deterministic verification step then downgrades any citation the analyst was not actually
shown.

**Handoff and synthesis.** Each worker hands control back via `transfer_back_to_supervisor`,
and the Supervisor relays their full analyses into one verdict, confidence score and 7-day
plan.

**Gate 2 - memory.** The system interrupts a second time before the irreversible write, then
saves the founder profile and this idea's verdict and key risk to long-term memory, keyed by
user rather than by thread - so the next idea is read in light of this one.

Every claim is traceable, every guess is labelled a guess, and the system is willing to say no.

## Submission requirements, and where each one is met

**Every claim in the write-up is backed by something visible in the file.** We hold ourselves
to that rule throughout: no statement in Section 8 or Section 9 rests on anything a reader cannot see in a
cell output below it. Section 10 re-checks the mechanical half of this list at runtime and prints the
result.

| Rubric section | Pts | Where it is met |
|----------------|-----|-----------------|
| 1. Agent fundamentals | 15 | Section 3 - two real tools that use every argument; every model call goes through `structured()` into a Pydantic contract (Section 1) |
| 2. Multi-agent / routing | 15 | Section 4.1 - LLM-emitted `transfer_to_*` handoff tool calls; Demo 3 exercises three routes |
| 3. RAG pipeline | 15 | Section 2 - load -> split -> embed -> store -> retrieve, plus the written Agentic-RAG justification |
| 4. Context and state | 15 | Section 4.5 Store (long-term) vs Section 5 checkpointer (short-term); T3 proves cross-thread |
| 5. Human-in-the-loop | 10 | Section 5 - two `interrupt()` gates before irreversible actions; Demos 1 and 2 execute both `interrupt()` and `Command(resume=...)` |
| 6. Functional API and errors | 15 | Section 4 `@task` / `@entrypoint`, no `StateGraph`; four error strategies, listed and located in Section 8 |
| 7. Workflow pattern | 10 | Section 5 - named **Orchestrator-Worker** and justified against Routing and Parallelization |
| 8. LangSmith | 5 | Section 0.1 sets `LANGCHAIN_TRACING_V2`; Section 8 write-up, backed by the config the same section prints |

| Preflight item | Where |
|----------------|-------|
| Full name in the notebook header | this cell |
| Track declared explicitly (A) | this cell, and re-checked in Section 10 |
| Programme name and cohort dates | this cell |
| Kernel restarted, all cells run top to bottom | run `Runtime > Restart session and run all` before submitting |
| Every demonstration cell has captured output | Section 6 Demos 1-3, Section 7 tests T1-T4 |
| `interrupt()` and `Command(resume=...)` both executed | Demo 1 (two resumes), Demo 2 (a resume that changes the result) |
| Cross-thread long-term memory demonstrated | T3 |
| No API keys in code or git history | Section 0.1 resolves from Colab Secrets; no key is ever written into a cell |
| No leftover template markers or placeholder text | verified - all cohort dates and project metadata filled |
| Write-up claims supported by visible output | Section 8 and Section 9, checked against the printed cells they cite |
| README and `.gitignore` | repository, outside this notebook - see Section 10 |

## Tests

| Test | Proves |
|------|--------|
| T1 | >=60% of findings are grounded in a verified source |
| T2 | A deliberately weak idea gets `no_go` or `pivot` - it can say no |
| T3 | A fact saved in thread A is read in thread B - real long-term memory |
| T4 | With the corpus emptied it degrades honestly instead of inventing |

## The four self-check questions the course asks

- *Does the retriever return actual answers?* Section 2.2 ends with a smoke test that prints the
  retrieved chunks, and a second one that proves the citation parser extracts their ids.
- *Is the routing LLM-based, not keyword matching?* There is no `if "market" in text` anywhere
  in this notebook. Section 4.1 prints the `transfer_to_*` tool calls the model itself emitted.
- *Does long-term memory persist across different thread IDs?* T3 writes in `t3-A` and reads
  in `t3-B`.
- *Is the tracing variable `LANGCHAIN_TRACING_V2`?* Yes - Section 0.1 sets it and prints it back.


## 0. Setup

In [ ]:
%pip install -qU langgraph langchain-groq langchain-community langchain-text-splitters
%pip install -qU langchain-huggingface sentence-transformers ddgs beautifulsoup4
print("installed - if Colab asks you to restart, restart and re-run from here")

### 0.1 Keys and tracing

Resolved in order: Colab Secrets -> environment -> typed prompt. Add them once in the Colab
sidebar (key icon) with Notebook access on.

| Secret | Required | Free at |
|--------|----------|---------|
| `GROQ_API_KEY` | yes | console.groq.com |
| `LANGSMITH_API_KEY` or `LANGCHAIN_API_KEY` | yes, for Section 8 | smith.langchain.com |
| `SERPER_API_KEY` | optional | serper.dev - skip it and search falls back to DuckDuckGo |

Tracing uses **`LANGCHAIN_TRACING_V2`**, not `LANGSMITH_TRACING_V2`. The wrong name produces
no trace and no error. No key is ever written into a cell, so nothing sensitive reaches git.

In [ ]:
import os, getpass

# Declared once, checked in Section 10.
COHORT = "23 Aug – 27 Aug 2026"

def _secret(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        return v.strip() if v and v.strip() else None
    except Exception:
        return None

def resolve(names, prompt=""):
    """Colab Secrets -> environment -> prompt. Returns (value, where_from)."""
    for n in names:
        if _secret(n):
            return _secret(n), f"Colab secret '{n}'"
    for n in names:
        if os.environ.get(n):
            return os.environ[n].strip(), f"environment '{n}'"
    if prompt:
        v = getpass.getpass(prompt).strip()
        if v:
            return v, f"Colab secret '{names[0]}'"
    return None, None

groq,   groq_src   = resolve(["GROQ_API_KEY"], "GROQ_API_KEY: ")
smith,  smith_src  = resolve(["LANGSMITH_API_KEY", "LANGCHAIN_API_KEY"], "LangSmith key: ")
serper, serper_src = resolve(["SERPER_API_KEY"], "SERPER_API_KEY (optional, Enter to skip): ")

if not groq:
    raise RuntimeError("GROQ_API_KEY is required.")
os.environ["GROQ_API_KEY"] = groq

if smith:
    project = _secret("LANGCHAIN_PROJECT") or "ideafoundry-capstone"
    for k, v in {"LANGCHAIN_API_KEY": smith, "LANGSMITH_API_KEY": smith,
                 "LANGCHAIN_TRACING_V2": "true", "LANGSMITH_TRACING": "true",
                 "LANGCHAIN_PROJECT": project, "LANGSMITH_PROJECT": project}.items():
        os.environ[k] = v
else:
    os.environ["LANGCHAIN_TRACING_V2"] = os.environ["LANGSMITH_TRACING"] = "false"

if serper:
    os.environ["SERPER_API_KEY"] = serper

# A rejected LangSmith key 403s on EVERY run and floods the notebook with warnings.
# Better to find out now, in one line, than to bury the real output.
if smith:
    import requests, logging
    endpoint = _secret("LANGCHAIN_ENDPOINT") or "https://api.smith.langchain.com"
    endpoints = [endpoint]
    if "eu" not in endpoint:
        endpoints.append("https://eu.api.smith.langchain.com")
    smith_ok = False
    detail = ""
    for ep in endpoints:
        try:
            probe = requests.get(f"{ep}/sessions", params={"limit": 1},
                                 headers={"x-api-key": smith}, timeout=15)
            if probe.status_code == 200:
                smith_ok = True
                endpoint = ep
                detail = "HTTP 200"
                break
            else:
                detail = f"HTTP {probe.status_code}"
        except Exception as e:
            detail = type(e).__name__

    if smith_ok:
        os.environ["LANGCHAIN_ENDPOINT"] = os.environ["LANGSMITH_ENDPOINT"] = endpoint
        os.environ["LANGCHAIN_TRACING_V2"] = os.environ["LANGSMITH_TRACING"] = "true"
        smith_src = f"{smith_src} - verified against {endpoint}"
    else:
        os.environ["LANGCHAIN_TRACING_V2"] = os.environ["LANGSMITH_TRACING"] = "false"
        logging.getLogger("langsmith.client").setLevel(logging.ERROR)
        smith_src = (f"KEY REJECTED ({detail}) - tracing disabled, section 8 scores 0. "
                     "Check smith.langchain.com/settings; if your account is in the EU "
                     "region add a LANGCHAIN_ENDPOINT secret set to "
                     "https://eu.api.smith.langchain.com")

print("project   :", "IdeaFoundry  |  Track A - Supervisor + Workers  |  Orchestrator-Worker")
print("cohort    :", COHORT)
print("groq     :", groq_src)
print("langsmith:", smith_src or "MISSING - tracing off, rubric section 8 scores 0")
print("serper   :", serper_src or "not set - using DuckDuckGo fallback")
print("tracing variable in use:", "LANGCHAIN_TRACING_V2 =",
      os.environ.get("LANGCHAIN_TRACING_V2"))


### 0.2 Imports and model

Groq's catalogue changes and access differs per account, so we ask the API what this key can
use instead of hardcoding an id that may 404. The free tier is 200k tokens/day **per model**,
so a spent model is not a broken one - it is out of budget until tomorrow. We probe each
candidate and pick two different models so the heavy stages and the plumbing draw on
separate buckets.

In [ ]:
import json, re, time, requests
from typing import Literal, Optional, List
from pydantic import BaseModel, Field, model_validator

from langchain_groq import ChatGroq
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langgraph.func import entrypoint, task
from langgraph.types import interrupt, Command, RetryPolicy
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.config import get_store

PREFERRED = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant", "llama-3.1-70b-versatile",
             "llama3-70b-8192", "llama3-8b-8192", "gemma2-9b-it", "qwen/qwen3.6-27b", "openai/gpt-oss-120b"]

_auth = {"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"}
_r = requests.get("https://api.groq.com/openai/v1/models", headers=_auth, timeout=30)
_r.raise_for_status()
AVAILABLE = {m["id"] for m in _r.json()["data"]}

def has_budget(model):
    """Send a test request. A 429 means this model's daily quota is spent."""
    try:
        r = requests.post("https://api.groq.com/openai/v1/chat/completions", headers=_auth,
                          json={"model": model, "max_tokens": 200,
                                "messages": [{"role": "user", "content": "hello"}]}, timeout=20)
        return r.status_code == 200, r.status_code
    except Exception as e:
        return False, type(e).__name__

usable = []
print(f"{'model':<30} daily budget")
for m in [x for x in PREFERRED if x in AVAILABLE]:
    ok, code = has_budget(m)
    print(f"{m:<30} {'OK' if ok else f'spent ({code})'}")
    if ok:
        usable.append(m)

if not usable:
    raise RuntimeError("Every model is rate-limited today. Wait for the daily reset or "
                       "upgrade at console.groq.com/settings/billing")

MODEL = usable[0]                                    # judgement: analysts, synthesis
SMALL = usable[1] if len(usable) > 1 else usable[0]  # plumbing: intake, routing, queries

llm = ChatGroq(model=MODEL, temperature=0)
llm_small = ChatGroq(model=SMALL, temperature=0)

print(f"\nmain model : {MODEL}")
print(f"small model: {SMALL}"
      + (" (separate daily bucket)" if SMALL != MODEL
         else " (same bucket - budget drains twice as fast)"))


## 1. Structured output - six contracts

Rubric Section 1. Everything the code reads is a parsed Pydantic model; nothing downstream parses
free text.

**`Claim` is the core design decision.** Every factual statement is wrapped with its
evidentiary basis, so "the AI said the market is growing" becomes either *grounded in
`setup_steps::0003`* or, honestly, *this is an assumption*.

The validators **coerce instead of raising**. A raising validator would blow up
`with_structured_output` mid-run on one model slip and destroy the whole analysis. An
unsupported claim quietly becoming a labelled assumption is both more robust and more honest.

One `WorkerAnalysis` schema serves all three analysts - they are interchangeable specialists
that differ only in role and tools (Section 4.2), so they do not need three near-identical contracts.

**Why the fields are narrow.** An earlier version widened them to `Union[float, str]` and
`Optional[List[...]]` to survive sloppy model output. That widening emitted `anyOf` and
`"type": ["array", "null"]` into the JSON schema, and Groq's strict tool-schema validation
rejects both - so every `WorkerAnalysis` call returned `BadRequestError` on
`function_calling` before silently falling back to `json_mode`. The schema now stays narrow
and all the forgiving coercion happens in a `mode="before"` validator, where it costs nothing
at the API boundary.

In [ ]:
import types
from typing import Union, get_origin, get_args

def _kinds(ann):
    """The base types inside an annotation, unwrapping Optional/Union."""
    if get_origin(ann) in (Union, getattr(types, "UnionType", Union)):
        out = set()
        for a in get_args(ann):
            out |= _kinds(a)
        return out
    return {get_origin(ann) or ann}

_TRUE, _FALSE = {"true", "yes", "1", "y"}, {"false", "no", "0", "n"}

def _to_float(v, default=0.5):
    try:
        return float(v)
    except (TypeError, ValueError):
        return default


class LLMSchema(BaseModel):
    """Base for anything the LLM fills in.

    Narrow types at the API boundary, forgiving coercion here. Two model slips are common and
    both come from smaller models: sending null for an empty list, and sending every parameter
    as a string ("True", "0.8"). This validator puts the real types back, so the rest of the
    code can assume bool is bool and float is float.
    """

    @model_validator(mode="before")
    @classmethod
    def _coerce(cls, data):
        if not isinstance(data, dict):
            return data
        out = dict(data)
        for name, f in cls.model_fields.items():
            if name not in out:
                continue
            v, k = out[name], _kinds(f.annotation)
            if v is None and list in k:                             # null -> []
                out[name] = []
            elif isinstance(v, str) and bool in k:                  # "True" -> True
                s = v.strip().lower()
                if s in _TRUE:
                    out[name] = True
                elif s in _FALSE:
                    out[name] = False
            elif isinstance(v, str) and (float in k or int in k):   # "0.8" -> 0.8
                out[name] = _to_float(v)
            elif v is None and (str in k or float in k or bool in k):
                out[name] = "" if str in k else (0.5 if float in k else True)
            elif list in k and not isinstance(v, list) and v is not None:
                out[name] = [v]                                     # scalar where a list belongs
        return out


class Claim(LLMSchema):
    """One factual statement with its evidentiary basis attached."""
    statement: str
    basis: Literal["evidence", "assumption"] = Field(
        default="assumption",
        description="'evidence' only if grounded in a source_id you were actually shown.")
    source_id: str = Field(
        default="", description="Corpus chunk id or URL, copied exactly. Empty for an assumption.")
    confidence: float = 0.5

    @model_validator(mode="after")
    def _needs_a_source(self):
        self.confidence = max(0.0, min(1.0, _to_float(self.confidence)))
        if self.basis == "evidence" and not self.source_id.strip():
            self.basis, self.confidence = "assumption", min(self.confidence, 0.4)
        if self.basis != "evidence":
            self.source_id = ""
        return self


class IdeaBrief(LLMSchema):
    """The Supervisor's structured reading of the idea. Shown to the founder at Gate 1."""
    one_liner: str
    sector: str = "unclear"
    geography: str = Field(default="Saudi Arabia",
                           description="City or country. Default to Saudi Arabia if unstated.")
    customer_segment: str = "unclear"
    business_model_guess: Literal["b2c", "b2b", "b2b2c", "marketplace", "unclear"] = "unclear"
    explicit_ask: str = Field(default="", description="What the founder asked for, in their terms.")
    is_analyzable: bool = Field(default=True,
                                description="False only if too vague to analyse at all.")


class RoutingDecision(LLMSchema):
    """The delegation decision. This is what makes it Track A.

    In the happy path this object is DERIVED from the supervisor's transfer_to_* tool calls
    (Section 4.1); it is also the fallback schema when tool calling itself fails.
    """
    route: Literal["analyze", "clarify"] = "analyze"
    workers: List[Literal["market", "competitor", "business"]] = Field(default_factory=list)
    rationale: str = ""
    clarifying_questions: List[str] = Field(default_factory=list)


class WorkerAnalysis(LLMSchema):
    """One analyst's output. Same shape for all three -- they differ by role and tools."""
    summary: str
    findings: List[Claim] = Field(default_factory=list)
    risks_or_gaps: List[str] = Field(default_factory=list)


class ApprovalDecision(LLMSchema):
    """What the founder sends back through Command(resume=...)."""
    approved: bool = True
    corrections: str = ""
    drop_workers: List[str] = Field(default_factory=list)
    redact_memory_keys: List[str] = Field(default_factory=list)


class FinalReport(LLMSchema):
    idea_one_liner: str
    market_opportunity: str
    competitors: str = ""
    business_model: str = ""
    setup_and_licensing: str = ""
    strengths: List[str] = Field(default_factory=list)
    weaknesses: List[str] = Field(default_factory=list)
    key_risks: List[str] = Field(default_factory=list)
    verdict: Literal["go", "conditional_go", "pivot", "no_go"] = "pivot"
    verdict_condition: str = Field(default="", description="Required if verdict is conditional_go.")
    what_would_change_our_mind: List[str] = Field(
        default_factory=list, description="2-3 findings that would flip the verdict.")
    next_7_days: List[str] = Field(
        default_factory=list, description="Concrete. Never 'do market research'.")
    overall_confidence: float = 0.5
    evidence_mode: Literal["grounded", "partial", "degraded"] = "partial"
    prior_idea_note: str = ""

    @model_validator(mode="after")
    def _clamp(self):
        self.overall_confidence = max(0.0, min(1.0, _to_float(self.overall_confidence)))
        return self


for _s in (Claim, IdeaBrief, RoutingDecision, WorkerAnalysis, ApprovalDecision, FinalReport):
    _js = json.dumps(_s.model_json_schema())
    print(f"{_s.__name__:<18} fields={len(_s.model_fields):<2} anyOf={_js.count('anyOf')}")
print("\n6 Pydantic schemas defined (data contracts, not language models)")

### 1.1 One structured-output call, with a model fallback

`with_structured_output` forces `tool_choice`, so when a model answers in prose instead of
calling the tool the API returns 400 *"Tool choice is required, but model did not call a
tool"* - and the good text it generated is thrown away. That is not a transient error, so
`RetryPolicy` cannot fix it. Two escapes, tried in order:

1. **`json_mode`** - ask for a raw JSON object instead of a tool call. Models that refuse to
   emit a tool call will usually still emit JSON.
2. **the other model** - a different model, a different failure mode.

That makes this **error strategy 4 of 4: retry on a different model** - for failures that
retrying the same model cannot fix. A spent daily quota is the other example: `RetryPolicy`
backs off for seconds, and the quota needs hours.

In [ ]:
def structured(schema, prompt, small=True):
    """Parse a model response into `schema`, cascading through fallbacks.

    The failure line prints the actual API message, not only the exception class. A bare
    `BadRequestError` with no text is what kept the schema bug of Section 1 invisible.
    """
    order = [llm_small, llm] if small else [llm, llm_small]
    plan, seen = [], set()
    for m in order:                      # de-duplicate when both names resolve to one model
        if id(m) not in seen:
            seen.add(id(m)); plan.append(m)

    schema_txt = json.dumps(schema.model_json_schema(), ensure_ascii=False)
    last = None
    for m in plan:
        for method in ("function_calling", "json_mode"):
            try:
                p = prompt if method == "function_calling" else (
                    prompt + "\n\nReturn ONLY a single JSON object matching this schema, "
                             "with no prose before or after:\n" + schema_txt)
                return m.with_structured_output(schema, method=method).invoke(p)
            except Exception as e:
                last = e
                if "429" in str(e) or "RateLimit" in type(e).__name__:
                    time.sleep(2.0)
                print(f"  [structured] {schema.__name__} via {method} on "
                      f"{getattr(m, 'model_name', '?')}: {type(e).__name__}: {str(e)[:200]}")
    raise last

print("structured() ready - reports WHY a call failed")


## 2. RAG pipeline

Rubric Section 3. All five stages - **load -> split -> embed -> store -> retrieve** - are below.

### Why Agentic RAG, not 2-Step or Hybrid

- **Query reformulation is mandatory.** *"A halal meal-prep delivery service for busy
  professionals in Riyadh"* is a product description, not a search string. It must become
  `commercial registration requirements`, `food establishment licence Saudi Arabia`.
- **Several retrievals per worker, each informed by the last.** The analyst finds that a
  municipal licence applies, then searches for what it requires.
- **The right to return nothing.** A 2-Step chain always stuffs its top-k into the prompt even
  when nothing relevant exists - which is exactly how ungrounded confidence gets laundered
  into a report. An agentic retriever can say *"the corpus does not cover this"*.

**We rejected Hybrid:** its only job here would be exact-matching licence names, which the
embedding model already handles at this corpus size.

### 2.1 Load - live, with a search fallback

The site's markup changes and Monsha'at is frequently unreachable from Colab, so this cell
prints how many characters it actually got per URL. If the primary source is thin we fall
back to live search **for the same material**, which keeps the pipeline genuinely grounded
with real URLs as provenance instead of pretending. Only if both fail does `CORPUS_OK` go
`False` and the system run degraded - it does not invent a corpus to cover a failed scrape.
That is honest failure, and it is T4.

**One namespace vocabulary.** The loader, the chunk ids, the tool docstring and the router all
use `setup_steps` / `business_planning`. An earlier version tagged the seed docs
`monshaat_steps` and the fallback docs `setup_steps`, so `search_corpus` filtered on a
namespace that matched zero chunks and every corpus search returned `NO_RESULTS` - which is
how the grounding rate reached 0% with no error anywhere.

In [ ]:
from bs4 import BeautifulSoup

NS_SETUP, NS_PLAN = "setup_steps", "business_planning"

SEED_URLS = [
    ("https://www.monshaat.gov.sa/ar/steps",           NS_SETUP),
    ("https://www.monshaat.gov.sa/en/toolkit-listing", NS_PLAN),
    ("https://www.monshaat.gov.sa/en/node/13287",      NS_SETUP),
    ("https://www.monshaat.gov.sa/en/node/13277",      NS_PLAN),
]
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; IdeaFoundry/1.0; capstone research)"}
MIN_CORPUS_CHARS = 4000

def fetch_clean(url, timeout=20):
    """Return (text, error). Never raises -- one dead URL must not kill the pipeline."""
    try:
        r = requests.get(url, headers=HEADERS, timeout=timeout)
        r.raise_for_status()
        r.encoding = r.apparent_encoding or "utf-8"
        soup = BeautifulSoup(r.text, "html.parser")
        for bad in soup(["script", "style", "nav", "footer", "header", "noscript", "svg"]):
            bad.decompose()
        lines = [ln.strip() for ln in soup.get_text("\n").split("\n") if len(ln.strip()) > 2]
        return "\n".join(lines), None
    except Exception as e:
        return "", f"{type(e).__name__}: {e}"

def web_results(query, k=5):
    """(url, title, snippet) triples from Serper if a key is set, else DuckDuckGo."""
    try:
        if os.environ.get("SERPER_API_KEY"):
            from langchain_community.utilities import GoogleSerperAPIWrapper
            raw = GoogleSerperAPIWrapper(k=k, gl="sa", hl="en").results(query)
            items = (raw or {}).get("organic", [])[:k]   # .get -- "organic" can be absent
            return [(i.get("link", ""), i.get("title", ""), i.get("snippet", "")) for i in items]
        from ddgs import DDGS
        items = list(DDGS().text(query, region="xa-en", max_results=k))
        return [(i.get("href", ""), i.get("title", ""), i.get("body", "")) for i in items]
    except Exception as e:
        print("  web search unavailable:", type(e).__name__)
        return []

FALLBACK_QUERIES = [
    ("how to start a business in Saudi Arabia commercial registration steps",   NS_SETUP),
    ("Saudi Arabia municipal licence requirements food establishment",          NS_SETUP),
    ("Saudi Arabia commercial registration MISA licence cost small business",   NS_SETUP),
    ("Monshaat business plan guide for small enterprises Saudi Arabia",         NS_PLAN),
    ("Saudi Arabia SME financing and support programmes Monshaat",              NS_PLAN),
    ("Saudi Arabia small business unit economics revenue model guide",          NS_PLAN),
]

raw_docs = []
print("--- primary source: Monsha'at ---")
print(f"{'chars':>8}  url")
for url, ns in SEED_URLS:
    text, err = fetch_clean(url)
    print(f"{len(text):>8}  {url}" + (f"\n          !! {err[:110]}" if err else ""))
    if len(text) > 200:
        raw_docs.append(Document(page_content=text,
                                 metadata={"source_url": url, "namespace": ns}))

TOTAL = sum(len(d.page_content) for d in raw_docs)

# Fall back PER NAMESPACE, so we never end up with a corpus that covers setup but not planning.
# A namespace with nothing in it is a silent failure that reads as "no evidence exists".
by_ns = {ns: sum(len(d.page_content) for d in raw_docs if d.metadata["namespace"] == ns)
         for ns in (NS_SETUP, NS_PLAN)}

if TOTAL < MIN_CORPUS_CHARS or min(by_ns.values()) < 1000:
    print(f"\n--- corpus thin ({TOTAL} chars, per-namespace {by_ns});"
          f" falling back to live search ---")
    seen_urls = {d.metadata["source_url"] for d in raw_docs}
    for query, ns in FALLBACK_QUERIES:
        if by_ns.get(ns, 0) >= 8000:              # this namespace is already well covered
            continue
        print(f"  query [{ns}]: {query}")
        for url, title, _snip in web_results(query, k=4):
            if not url or url in seen_urls:
                continue
            seen_urls.add(url)
            text, err = fetch_clean(url)
            print(f"{len(text):>8}  {url}" + (f"  !! {err[:70]}" if err else ""))
            if len(text) > 300:
                raw_docs.append(Document(page_content=text,
                                         metadata={"source_url": url, "namespace": ns}))
                by_ns[ns] = by_ns.get(ns, 0) + len(text)
    TOTAL = sum(len(d.page_content) for d in raw_docs)

CORPUS_OK = TOTAL >= MIN_CORPUS_CHARS
print(f"\ntotal {TOTAL} chars in {len(raw_docs)} docs   per-namespace: {by_ns}")
print(f"CORPUS_OK = {CORPUS_OK}")
if not CORPUS_OK:
    print(">> DEGRADED mode: every claim will be an assumption. The system will say so")
    print(">> in the report rather than inventing grounding.")

### 2.2 Split, embed, store, retrieve

Chunk ids are stable and human-readable (`setup_steps::0003`) because they are what
`Claim.source_id` points at and what the founder uses to check our work.

**`verify_claims()` is our anti-hallucination gate:** any `source_id` that was not actually
offered to *that* analyst gets its claim downgraded to an assumption. Without it, "grounding
rate" would only measure how confidently the model invents citations.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=900, chunk_overlap=120,
    separators=["\n\n", "\n", ". ", "، ", " ", ""])      # Arabic comma included

chunks, counters = [], {}
for doc in raw_docs:
    for piece in splitter.split_documents([doc]):
        ns = piece.metadata["namespace"]
        counters[ns] = counters.get(ns, 0) + 1
        piece.metadata["chunk_id"] = f"{ns}::{counters[ns]:04d}"
        chunks.append(piece)
CHUNK_IDS = {c.metadata["chunk_id"] for c in chunks}

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = InMemoryVectorStore(embedding=embeddings)
if chunks:
    vector_store.add_documents(chunks)

def retrieve(query, namespace=None, k=4):
    """Real retrieval. Uses every argument it is given.

    A namespace filter that matches nothing falls back to an unfiltered search instead of
    returning []. A label the model guessed slightly wrong should cost relevance, never the
    whole result set.

    NOTE: InMemoryVectorStore's `filter` is a CALLABLE, not a dict.
    """
    if not chunks:
        return []
    hits = []
    if namespace:
        flt = lambda d: d.metadata.get("namespace") == namespace
        hits = vector_store.similarity_search(query, k=k, filter=flt)
    if not hits:
        hits = vector_store.similarity_search(query, k=k)
    return [{"chunk_id": h.metadata["chunk_id"], "text": h.page_content} for h in hits]


# An earlier version wrote this as r"\\[source_id:\\s*(.+?)\\]". In a raw string `\\[` is a
# LITERAL BACKSLASH followed by `[`, so the pattern compiled to "a backslash, then one character
# from the class [source_id:\s*(.+?)\]" -- with the capture group swallowed into the character
# class. It matched nothing, ever, so offered_ids() always returned an empty set and
# verify_claims() downgraded 100% of findings to assumptions.
SOURCE_RE = re.compile(r"\[source_id:\s*([^\]\n]+)\]")

def offered_ids(evidence: str) -> set:
    """Exactly the source ids this analyst was actually shown."""
    return {m.strip() for m in SOURCE_RE.findall(evidence or "") if m.strip()}

def verify_claims(claims, allowed=None):
    """Downgrade any claim whose source_id was not actually offered to that analyst.

    Matching is case- and whitespace-tolerant and ignores a trailing slash on a URL, so a
    cosmetic slip does not destroy a genuinely grounded finding -- but anything that cannot be
    tied to a real offered id becomes an honest assumption.
    """
    allowed = CHUNK_IDS if allowed is None else set(allowed)
    lookup = {}
    for a in allowed:
        lookup[a.strip().lower()] = a
        lookup[a.strip().lower().rstrip("/")] = a
    out = []
    for c in claims or []:
        if c.basis != "evidence":
            out.append(c); continue
        key = (c.source_id or "").strip().lower()
        real = lookup.get(key) or lookup.get(key.rstrip("/"))
        out.append(c.model_copy(update={"source_id": real}) if real else
                   c.model_copy(update={"basis": "assumption", "source_id": "",
                                        "confidence": min(_to_float(c.confidence), 0.4)}))
    return out

print(f"{len(chunks)} chunks: {counters}")

# SMOKE TEST -- the rubric's own check for a dead retriever, plus the citation parser.
hits = retrieve("steps to start a business and register a commercial licence", k=2)
print("\nsmoke test retrieve  :", "OK" if hits else ">> RETRIEVER RETURNED NOTHING - fix 2.1")
for h in hits:
    print(f"  [{h['chunk_id']}] {h['text'][:160]}...".replace("\n", " "))

_probe = "\n\n".join(f"[source_id: {h['chunk_id']}]\n{h['text'][:80]}" for h in hits)
print("smoke test offered_ids:",
      offered_ids(_probe) or ">> SOURCE_RE IS BROKEN - grounding will read 0%")

## 3. Tools and the agentic loop

Rubric Section 1. Neither tool returns a hardcoded string, and both use every argument. If a function
ignores its arguments it is not a tool call - it is decoration.

`web_search` exists because a static corpus can never name competitors. The corpus is
authoritative on licensing and business planning; it has never heard of your rivals.

`run_tools` is where "agentic" actually happens: the tools are bound to the model, it writes
its own search queries and decides whether to search again. The returned `calls` list records
what it chose, so this is inspectable rather than a claim.

In [ ]:
@tool
def search_corpus(query: str, namespace: str = "", k: int = 4) -> str:
    """Search official Saudi business guidance on setup, licensing, registration, financing and
    business planning. namespace='setup_steps' for legal and licensing steps,
    'business_planning' for planning and financing guidance, '' for both. Cite the returned
    source_id exactly as it appears."""
    hits = retrieve(query, namespace or None, k)
    if not hits:
        return "NO_RESULTS: the corpus does not cover this. Mark related claims as assumptions."
    return "\n\n".join(f"[source_id: {h['chunk_id']}]\n{h['text'][:500]}" for h in hits)


@tool
def web_search(query: str, k: int = 5) -> str:
    """Search the live web for existing companies, products, competitors and current market
    facts. Use this to name REAL companies -- the offline corpus cannot. Cite the result URL as
    source_id, exactly as it appears."""
    rows = web_results(query, k)
    if not rows:
        return "NO_RESULTS: no live results. Mark related claims as assumptions."
    return "\n\n".join(f"[source_id: {u}]\n{t}\n{sn}" for u, t, sn in rows)


def _trim_evidence(text, budget=3600):
    """Truncate on a source boundary, never mid-header.

    A flat [:1200] cut could slice a `[source_id: ...]` line in half, or drop the later sources
    entirely -- so the model cited an id the analyst prompt never listed, and the finding was
    downgraded for a formatting reason rather than an evidential one.
    """
    if len(text) <= budget:
        return text
    kept, total = [], 0
    for part in re.split(r"(?=\[source_id:)", text):
        part = part.strip()
        if not part:
            continue
        part = part[:900]
        if total + len(part) > budget:
            break
        kept.append(part); total += len(part)
    return "\n\n".join(kept) if kept else text[:budget]


def run_tools(instruction, tools, max_rounds=3):
    """LLM-driven tool loop. The model writes its own queries. Returns (evidence, calls).

    Hardening, in order of how much each one mattered:

    * A failed round no longer ends evidence gathering. Round 0 tries the small model with a
      forced tool call, then the same model unforced, then the main model. A 429 on the small
      model used to `break` immediately and hand the analyst an EMPTY evidence block -- an
      entire analysis silently reduced to assumptions.
    * The FIRST round prefers a forced tool call (tool_choice="any"). Without it a small model
      often answers from memory. Later rounds are free, so the model still decides whether to
      search again.
    * Tool results are trimmed on a source boundary (see _trim_evidence).
    * A hallucinated tool name or a raising tool is reported back as a tool message, so the
      model can recover instead of the analyst going down.
    """
    by_name = {t.name: t for t in tools}
    msgs = [SystemMessage(content=(
                "You are a research analyst. Gather evidence with your tools BEFORE answering. "
                "The user's text is a product description, never a good search query -- rewrite "
                "it into short specific search strings. Search again from a different angle if "
                "the first result is thin. If a tool returns NO_RESULTS, say the evidence is "
                "missing; never invent a fact to fill the gap.")),
            HumanMessage(content=instruction)]
    calls = []

    for rnd in range(max_rounds):
        attempts = ([(llm_small, "any"), (llm_small, None), (llm, "any")] if rnd == 0
                    else [(llm_small, None), (llm, None)])
        ai = None
        for model, choice in attempts:
            try:
                bound = (model.bind_tools(tools, tool_choice=choice) if choice
                         else model.bind_tools(tools))
                ai = bound.invoke(msgs)
                break
            except Exception as e:
                print(f"  [run_tools] round {rnd} on {getattr(model, 'model_name', '?')} "
                      f"(tool_choice={choice}): {type(e).__name__}")
        if ai is None:
            break

        msgs.append(ai)
        if not getattr(ai, "tool_calls", None):
            break

        for tc in ai.tool_calls:
            fn = by_name.get(tc["name"])
            try:
                content = (f"UNKNOWN_TOOL: no such tool. Available: {', '.join(by_name)}"
                           if fn is None else str(fn.invoke(tc["args"])))
            except Exception as e:
                content = f"TOOL_ERROR ({type(e).__name__}). Mark related claims as assumptions."
            calls.append({"tool": tc["name"], "args": tc["args"]})
            msgs.append(ToolMessage(content=_trim_evidence(content), tool_call_id=tc["id"]))

    evidence = "\n\n".join(m.content for m in msgs if isinstance(m, ToolMessage))
    return evidence, calls


_ev, _calls = run_tools("Idea: halal meal-prep delivery subscription in Riyadh. Find the Saudi "
                        "licensing and registration requirements.", [search_corpus])
print("queries the LLM wrote itself:")
for c in _calls:
    print("  ", c["tool"], c["args"])
print(f"\n{len(_ev)} chars of evidence gathered")
print("source ids offered:", sorted(offered_ids(_ev))[:6] or ">> STILL ZERO - re-run 2.1 and 2.2")

## 4. The agents

Rubric Section 6 - built with `@task` and `@entrypoint`, **not** `StateGraph`.

**A note on the Track A reference implementation.** The lesson demonstrates handoffs with
`langgraph_supervisor.create_supervisor(...)`, which compiles a `StateGraph`. The capstone
rubric Section 6 requires the Functional API and marks `StateGraph` as a mistake, so the two cannot
both be satisfied by copying the reference. We keep the Functional API and reproduce the
handoff *semantics* directly: the supervisor emits real `transfer_to_*` tool
calls (Section 4.1), each worker gets only its own tools and never sees another worker's output
(Section 4.2), and every worker hands control back to the supervisor, which relays the full result
(Section 4.3). We print the `transfer_to_*` calls as direct evidence that the **LLM** chose the
workers.

**Reliability 1 of 4 - `RetryPolicy`.** Groq returns 429/503 under load, which is exactly what
three concurrent workers create. Every LLM task carries a real policy object. No loops, no
`time.sleep()`.

**Task boundaries speak plain dicts.** Every task argument and return value is checkpointed,
and LangGraph warns on deserialising unregistered classes - so tasks exchange
`.model_dump()` dicts and the models are rebuilt where attribute access is needed.

In [ ]:
RETRY = RetryPolicy(max_attempts=3, initial_interval=1.0, backoff_factor=2.0,
                    jitter=True, retry_on=(Exception,))

@task(retry_policy=RETRY)
def intake(raw_idea: str) -> dict:
    """Supervisor stage 0: free text -> structured brief."""
    return structured(IdeaBrief, f"""You are the supervisor of an idea-analysis team for founders in Saudi Arabia.
Extract a structured brief. Assume Saudi Arabia if geography is unstated. Set
is_analyzable=false ONLY if the idea is too vague to analyse at all (e.g. "an app",
"something with AI"); a specific idea missing minor detail IS analyzable.

RAW IDEA:
{raw_idea}""").model_dump()

### 4.1 Routing by handoff - the decision that makes this Track A

Rubric Section 2. Keyword matching is not routing - there is no `if "market" in text` anywhere here.

We give the supervisor four **handoff tools** and ask it to call the ones it needs. The
`transfer_to_*` tool calls the model emits **are** the routing decision; a `RoutingDecision`
is derived from them so the rest of the pipeline still reads a validated Pydantic contract.
We print the calls at routing time and again at Gate 1, which is direct evidence that the LLM,
not a rule, picked the workers.

Four outcomes are genuinely reachable: `request_clarification` with no workers, competitor
only, business only, or all three. **The clarify branch matters most:** it is the only route
that produces no report, which proves the router is deciding rather than dispatching. Demo 3
exercises three of the four.

If the model emits no tool call at all, we fall back to a structured `RoutingDecision` rather
than failing the run - and if that also comes back empty on an `analyze` route, the full panel
runs. A supervisor that forgot to fill one field is not a reason to refuse to analyse.

In [ ]:
@tool
def transfer_to_market(reason: str) -> str:
    """Hand off to the Market Analyst. Use when the founder needs the customer problem, demand
    signals, or the size and shape of the opportunity assessed."""
    return f"HANDOFF ACCEPTED by market analyst. Reason: {reason}"

@tool
def transfer_to_competitor(reason: str) -> str:
    """Hand off to the Competitor Analyst. Use when the founder needs real existing companies
    and substitutes named, and the gap between them identified."""
    return f"HANDOFF ACCEPTED by competitor analyst. Reason: {reason}"

@tool
def transfer_to_business(reason: str) -> str:
    """Hand off to the Business Analyst. Use when the founder needs revenue models, unit
    economics, or the concrete Saudi licensing and setup requirements."""
    return f"HANDOFF ACCEPTED by business analyst. Reason: {reason}"

@tool
def request_clarification(questions: List[str], reason: str) -> str:
    """Do NOT hand off to any analyst; ask the founder for more detail first. Use only when the
    idea is too vague to analyse at all. Provide 2-4 specific questions."""
    return f"NO HANDOFF. Clarification requested: {questions}. Reason: {reason}"

HANDOFF_TOOLS = [transfer_to_market, transfer_to_competitor,
                 transfer_to_business, request_clarification]
HANDOFF_BY_NAME = {t.name: t for t in HANDOFF_TOOLS}
HANDOFF_TO_WORKER = {"transfer_to_market": "market",
                     "transfer_to_competitor": "competitor",
                     "transfer_to_business": "business"}


def history_text(prior):
    return "\n".join(f"- {p['idea_summary']} -> {p['verdict']} (risk: {p['key_risk']})"
                     for p in prior) or "none on record"


ROUTING_BRIEF = """You are the supervisor of an idea-analysis team for founders in Saudi Arabia.
Delegate this idea by calling the handoff tools for the analysts it actually needs.

- transfer_to_market     : customer problem, demand signals, opportunity in this geography
- transfer_to_competitor : names real existing companies, using live web search
- transfer_to_business   : revenue models, unit economics, Saudi licensing and setup
- request_clarification  : the idea is too vague to analyse at all

Rules:
- is_analyzable false -> call request_clarification ONLY, with 2-4 questions.
- If explicit_ask is narrow, hand off ONLY to the analysts that ask needs. Running all three on
  a narrow question wastes the founder's time and money.
- All three only for a genuine full-validation request.
- Give a one-sentence reason with every handoff.

PRIOR IDEAS FROM THIS FOUNDER:
{history}

BRIEF:
{brief}"""


@task(retry_policy=RETRY)
def route(brief: dict, prior: list) -> dict:
    """Supervisor stage 1: delegate through explicit handoff TOOL CALLS.

    Returns {"decision": RoutingDecision dict, "handoffs": [...], "via": "tool_calls"|"structured"}.
    """
    prompt = ROUTING_BRIEF.format(history=history_text(prior),
                                  brief=json.dumps(brief, ensure_ascii=False, indent=2))
    handoffs, tcs = [], []
    for model, choice in [(llm_small, "any"), (llm_small, None), (llm, "any")]:
        try:
            bound = (model.bind_tools(HANDOFF_TOOLS, tool_choice=choice) if choice
                     else model.bind_tools(HANDOFF_TOOLS))
            ai = bound.invoke([SystemMessage(content="You delegate by calling handoff tools."),
                               HumanMessage(content=prompt)])
            tcs = getattr(ai, "tool_calls", None) or []
            if tcs:
                break
        except Exception as e:
            print(f"  [route] handoff attempt on {getattr(model, 'model_name', '?')} "
                  f"(tool_choice={choice}): {type(e).__name__}")

    questions = []
    for tc in tcs:
        name, args = tc["name"], (tc.get("args") or {})
        reason = str(args.get("reason", ""))
        if name == "request_clarification":
            qs = args.get("questions") or []
            questions += [str(q) for q in (qs if isinstance(qs, list) else [qs])]
            handoffs.append({"from": "supervisor", "to": "founder", "via": name,
                             "reason": reason, "result": str(request_clarification.invoke(args))[:160]})
        elif name in HANDOFF_TO_WORKER:
            # Actually invoke the handoff tool, so the ledger records the tool's own reply --
            # the same "HANDOFF ACCEPTED" acknowledgement a worker agent would send back.
            handoffs.append({"from": "supervisor", "to": HANDOFF_TO_WORKER[name], "via": name,
                             "reason": reason,
                             "result": str(HANDOFF_BY_NAME[name].invoke(args))[:160]})

    workers = [h["to"] for h in handoffs if h["to"] in HANDOFF_TO_WORKER.values()]
    if handoffs:
        decision = RoutingDecision(
            route="analyze" if workers else "clarify",
            workers=sorted(set(workers), key=workers.index),
            rationale=" | ".join(f"{h['via']}: {h['reason']}" for h in handoffs) or "handoff",
            clarifying_questions=questions)
        via = "tool_calls"
    else:
        # Fallback: no tool call came back at all. Decide with a structured contract instead of
        # failing the run.
        print("  [route] no handoff tool call -- falling back to structured RoutingDecision")
        decision = RoutingDecision(**structured(RoutingDecision, prompt).model_dump())
        via = "structured"

    print(f"   supervisor delegation via {via}:")
    for h in handoffs or [{"via": "(structured)", "to": ",".join(decision.workers) or "clarify",
                           "reason": decision.rationale}]:
        print(f"     {h['via']:<24} -> {h['to']:<11} :: {str(h['reason'])[:70]}")

    return {"decision": decision.model_dump(), "handoffs": handoffs, "via": via}

### 4.2 The analysts - one function, three roles

The three workers are interchangeable specialists that differ only in role, tools and
evidence, so they are one `@task` driven by a spec table rather than three near-identical
functions. **Each sees only the brief and its own tools - no worker ever sees another
worker's output**, which is the invariant that makes this Supervisor + Workers.

Note the honest division of grounding. The Competitor analyst relies on live search because no
static corpus has ever been able to name a competitor. We give the Market analyst both tools:
an earlier version had it on the corpus alone, and since the corpus is authoritative on
licensing rather than on demand, it had nothing it could legitimately cite and produced zero
grounded findings by construction.

In [ ]:
WORKERS = {
    "market": dict(
        role="Market Analyst",
        job="Assess the customer problem, the target audience, and the size and shape of the "
            "opportunity in this geography.",
        tools=[search_corpus, web_search]),
    "competitor": dict(
        role="Competitor Analyst",
        job="Name REAL existing companies and substitutes serving this customer, and where the "
            "gap is. Never invent a company name; if the evidence names none, say so.",
        tools=[web_search]),
    "business": dict(
        role="Business Analyst",
        job="Cover revenue models, unit economics, the concrete Saudi licensing and setup "
            "requirements, and the risks that kill this business.",
        tools=[search_corpus, web_search]),
}


@task(retry_policy=RETRY)
def analyst(name: str, brief: dict, degraded: bool) -> dict:
    """One worker. Same code for all three -- only role, tools and evidence differ."""
    spec = WORKERS[name]
    started = time.time()
    tool_names = {t.name for t in spec["tools"]}
    offline_only = tool_names == {"search_corpus"}

    # Evidence gathering is the fragile half. If it fails, the analyst should still deliver an
    # honest ungrounded analysis rather than take the whole section down with it.
    evidence, calls, note = "", [], ""
    if degraded and offline_only:
        note = "The corpus is UNAVAILABLE. Every finding must use basis='assumption'."
    else:
        try:
            evidence, calls = run_tools(
                f"{spec['job']}\n\nIdea: {json.dumps(brief, ensure_ascii=False)}", spec["tools"])
        except Exception as e:
            print(f"  [{name}] evidence gathering failed: {type(e).__name__}: {e}")
        note = ("If a statement is not supported by the evidence below, use basis='assumption' "
                "-- that is the correct answer, not a failure.")

    allowed = offered_ids(evidence)
    id_list = "\n".join(f"- {i}" for i in sorted(allowed)[:40]) or "(no sources available)"

    # The citation instructions are deliberately short. An earlier version spent six lines
    # threatening the model about invented ids; that pressure does not improve citation
    # accuracy, it just crowds out the actual task. The guarantee comes from verify_claims(),
    # which is deterministic -- so the prompt only has to be clear, not defensive.
    out = structured(WorkerAnalysis, f"""You are the {spec['role']}. {spec['job']}

Return 4-8 findings.
Use basis="evidence" and copy the source_id EXACTLY from this list whenever the evidence
below supports the statement:
{id_list}
Otherwise use basis="assumption" with an empty source_id. {note}

BRIEF:
{json.dumps(brief, ensure_ascii=False, indent=2)}

EVIDENCE:
{evidence or '(none available)'}""", small=False)

    out.findings = verify_claims(out.findings, allowed)
    grounded = sum(1 for c in out.findings if c.basis == "evidence")
    seconds = round(time.time() - started, 1)
    print(f"   [{name}] {len(calls)} tool call(s), {len(allowed)} source(s) offered, "
          f"{grounded}/{len(out.findings)} findings grounded, {seconds}s"
          f"  -> transfer_back_to_supervisor")
    # `_seconds` is bookkeeping, not analysis. Section 8 claims the fan-out is genuinely concurrent, so
    # we measure it rather than asserting it -- summing these against the orchestrator's own wall
    # clock is what makes that claim checkable inside this file.
    payload = out.model_dump()
    payload["_seconds"] = seconds
    return payload

print("one analyst task, three roles:", list(WORKERS))

### 4.3 Synthesis - and the other two reliability strategies

**Reliability 2 of 4 - fallback to degraded mode.** With no corpus, every corpus claim becomes
an assumption, confidence is capped at 0.35 by deterministic code rather than by asking the
model nicely, and the report says so plainly. Without this, an outage turns IdeaFoundry
into the confident-fiction machine it exists to replace. This is T4.

**Reliability 3 of 4 - graceful partial.** A worker that still fails after its retries has its
section marked unavailable and confidence lowered, rather than discarding the analyses that
succeeded.

(Strategy 4 of 4, *retry on a different model*, lives in `structured()` in Section 1.1.)

This is also where the supervisor **relays the workers' full answers**: every returned
`WorkerAnalysis` is passed into the synthesis prompt verbatim, not summarised on the way in.

In [ ]:
@task(retry_policy=RETRY)
def synthesize(brief: dict, parts: dict, prior: list, degraded: bool, failed: list) -> dict:
    """Supervisor stage 4: merge the analysts into one defensible verdict."""
    def _clean(d):
        return {k: v for k, v in d.items() if not k.startswith("_")}

    blocks = [f"### {n}\n" + ("STATUS: UNAVAILABLE (failed after retries)." if n in failed
                              else json.dumps(_clean(parts[n]), ensure_ascii=False, indent=2))
              for n in list(parts) + failed]

    guard = []
    if degraded:
        guard.append("EVIDENCE MODE IS DEGRADED: the corpus was unavailable. Set "
                     "evidence_mode='degraded', cap overall_confidence at 0.35, and say plainly "
                     "in market_opportunity that this analysis is ungrounded.")
    if failed:
        guard.append(f"These analysts produced nothing: {failed}. Say so and lower confidence.")

    report = structured(FinalReport, f"""You are the supervisor writing the final evaluation. You are an analyst, not a
cheerleader.

- Be willing to return 'no_go' or 'pivot'. Most early ideas deserve one. A saturated market
  entered with no differentiation is a no_go, not a conditional_go.
- 'conditional_go' REQUIRES a specific named condition in verdict_condition.
- what_would_change_our_mind: 2-3 findings that would genuinely flip the verdict.
- next_7_days: concrete and doable in a week ("interview 5 restaurant owners in Al Malaz about
  delivery margins"), never "conduct market research".
- Where analysts disagree, say so. Do not average them into mush.
- If this idea shares a risk with a prior idea below, say so in prior_idea_note.
{chr(10).join(guard)}

PRIOR IDEAS: {history_text(prior)}

BRIEF:
{json.dumps(brief, ensure_ascii=False, indent=2)}

ANALYST REPORTS:
{chr(10).join(blocks)}""", small=False)

    # Deterministic guards. Never trust a model to obey a cap it was merely asked to obey.
    if degraded:
        report.evidence_mode = "degraded"
        report.overall_confidence = min(report.overall_confidence, 0.35)
    if failed:
        report.overall_confidence = min(report.overall_confidence, 0.5)
    if report.verdict == "conditional_go" and not report.verdict_condition.strip():
        report.verdict = "pivot"                  # an unnamed condition is not a condition
    return report.model_dump()

print("synthesize ready")

### 4.4 Watch RetryPolicy fire

Rubric Section 6 wants error handling that is real, not described. This task fails twice and succeeds
on the third attempt - the policy object handles it, with no loop and no `sleep` in the body.
Open the LangSmith trace afterwards and you will see three attempts.

In [ ]:
_n = {"i": 0}

@task(retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.4, backoff_factor=2.0,
                               jitter=True, retry_on=(ConnectionError,)))
def flaky_probe(label: str) -> str:
    """Simulates a transient Groq 429. Fails twice, then succeeds."""
    _n["i"] += 1
    print(f"  attempt {_n['i']}")
    if _n["i"] < 3:
        raise ConnectionError("simulated transient upstream error (429)")
    return f"{label}: succeeded on attempt {_n['i']}"

@entrypoint(checkpointer=InMemorySaver())
def retry_demo(label: str) -> str:
    return flaky_probe(label).result()

print(retry_demo.invoke("rate-limit probe", {"configurable": {"thread_id": "retry"}}))

### 4.5 Long-term memory - a separate Store

Rubric Section 4. Two distinct mechanisms:

- **Short-term state = the checkpointer**, keyed by `thread_id`. It holds one session,
  including the paused state at each `interrupt()`. A growing list of chat messages is this -
  it is not long-term memory, and the rubric rules that out explicitly.
- **Long-term memory = the Store**, keyed by `user_id` under `("founder", user_id)`. It
  survives across threads.

**Why it earns its place.** Founders pitch adjacent ideas repeatedly. When idea #2 arrives in a
sector where idea #1 was rejected on a distribution risk, the Supervisor should say so. Nothing
without cross-thread memory can. T3 proves it.

In [ ]:
def load_profile(store, uid):
    item = store.get(("founder", uid), "profile")
    return item.value if item else {}

def load_history(store, uid):
    item = store.get(("founder", uid), "idea_history")
    return item.value.get("ideas", []) if item else []

@task
def memory_proposal(brief: dict, report: dict) -> dict:
    """What we PROPOSE to remember. Shown to the founder at Gate 2 before any write."""
    return {"profile": {"sectors_of_interest": [brief["sector"]],
                        "usual_geography": brief["geography"]},
            "idea_entry": {"idea_summary": brief["one_liner"], "verdict": report["verdict"],
                           "key_risk": (report["key_risks"] or ["unspecified"])[0]}}

@task
def persist_memory(uid: str, proposal: dict, redact: list) -> dict:
    """The irreversible write that Gate 2 guards."""
    store = get_store()
    if "profile" not in redact:
        old = load_profile(store, uid)
        sectors = sorted(set(old.get("sectors_of_interest", []))
                         | set(proposal["profile"]["sectors_of_interest"]))
        store.put(("founder", uid), "profile",
                  {**old, **proposal["profile"], "sectors_of_interest": sectors})
    if "idea_entry" not in redact:
        ideas = load_history(store, uid) + [proposal["idea_entry"]]
        store.put(("founder", uid), "idea_history", {"ideas": ideas})
    return {"written": [k for k in ("profile", "idea_entry") if k not in redact],
            "redacted": redact}

print("memory helpers ready")

## 5. The entrypoint - Orchestrator-Worker with two gates

Rubric Section 5, Section 6, Section 7.

**We name the pattern: Orchestrator-Worker.** The Supervisor decomposes the request into worker
assignments, dispatches them via handoff tool calls, and synthesises the results.
Parallelization happens *inside* the pattern - the fan-out resolves concurrently - but the
governing structure is Orchestrator-Worker, because the worker set is chosen at runtime by
the orchestrator rather than being a fixed split. Routing is a component of the same
decision, not a separate top-level pattern: the router picks *who works*, and the
orchestrator still owns decomposition and synthesis.

**Two gates, both guarding something irreversible.** The rubric asks for `interrupt()` before
an irreversible action; showing a report is reversible, so "approve the report" would not
qualify.

- **Gate 0 - clarification, only when needed.** A vague idea pauses for answers instead of
  being rejected, then is re-read and re-routed. Bounded to one round.
- **Gate 1 - before spending on analysis.** Worker runs cost tokens, and analysis built on a
  misread brief is wasted work. Corrections here change the run (Demo 2).
- **Gate 2 - before writing to permanent memory.** This data shapes every future verdict.
  Genuinely irreversible, and the right privacy posture.

On resume, the entrypoint body replays from the top. Completed `@task` results are replayed
from the checkpoint rather than re-run - which is why every LLM call and every write lives
inside a `@task`.

In [ ]:
checkpointer = InMemorySaver()
store = InMemoryStore()

ALL_WORKERS = ["market", "competitor", "business"]

def _approval(x):
    """Accept whatever the founder resumed with, without crashing on the shape."""
    if isinstance(x, ApprovalDecision):
        return x
    if isinstance(x, dict):
        return ApprovalDecision(**x)
    return ApprovalDecision(approved=bool(x))


@entrypoint(checkpointer=checkpointer, store=store)
def analyze_idea(payload: dict) -> dict:
    """Orchestrator-Worker. payload = {"idea": str, "user_id": str}"""
    raw_idea, uid = payload["idea"], payload.get("user_id", "founder-1")
    mem = get_store()
    prior = load_history(mem, uid)
    profile = load_profile(mem, uid)

    # --- clarification loop: a vague idea is not a dead end ---------------
    # The state is already checkpointed, so we can pause, take the answers, fold them into the
    # idea and re-read it. Bounded to one round so a founder who cannot answer still gets a
    # clean exit instead of an infinite loop.
    text = raw_idea
    for attempt in range(2):
        brief = intake(text).result()
        routed = route(brief, prior).result()
        decision = RoutingDecision(**routed["decision"])
        handoffs = routed["handoffs"]
        if decision.route != "clarify" or attempt == 1:
            break
        answers = interrupt({"gate": "clarification",
                             "question": "This idea is too vague to analyse. Answer these and "
                                         "I will re-read it.",
                             "brief": brief, "questions": decision.clarifying_questions,
                             "rationale": decision.rationale, "handoffs": handoffs,
                             "reply_with": "Command(resume={'answers': str})  "
                                           "-- send an empty string to stop here"})
        answers = (answers or {}).get("answers", "") if isinstance(answers, dict) else str(answers)
        if not str(answers).strip():
            return {"status": "needs_clarification", "brief": brief, "handoffs": handoffs,
                    "questions": decision.clarifying_questions,
                    "rationale": decision.rationale}
        text = f"{raw_idea}\n\nFOUNDER'S ANSWERS TO THE CLARIFYING QUESTIONS:\n{answers}"

    if decision.route == "clarify":
        return {"status": "needs_clarification", "brief": brief, "handoffs": handoffs,
                "questions": decision.clarifying_questions,
                "rationale": decision.rationale + " (still too vague after one clarification)"}

    # ===== GATE 1: interrupt() before spending on the workers =============
    ok = _approval(interrupt({
        "gate": "analysis_plan",
        "question": "Is this reading correct, and should these analysts run?",
        "brief": brief, "route": decision.route, "workers": decision.workers,
        "handoffs": handoffs, "routed_via": routed["via"],
        "rationale": decision.rationale,
        "clarifying_questions": decision.clarifying_questions,
        "known_profile": profile, "known_prior_ideas": prior,
        "reply_with": "Command(resume={'approved': bool, 'corrections': str, "
                      "'drop_workers': [str]})"}))
    if not ok.approved:
        return {"status": "cancelled_by_founder", "brief": brief, "handoffs": handoffs}

    # A correction RE-DERIVES the brief, so it genuinely changes the analysis.
    if ok.corrections.strip():
        brief = intake(f"{text}\n\nFOUNDER CORRECTION (authoritative): "
                       f"{ok.corrections}").result()

    # An analyze route with an empty worker list is a supervisor that forgot to fill one field,
    # not a reason to refuse to analyse. Fall back to the full panel and carry on.
    assigned = decision.workers or ALL_WORKERS
    workers = [w for w in assigned if w not in ok.drop_workers]
    for w in ok.drop_workers:
        if w in assigned:
            handoffs.append({"from": "founder", "to": w, "via": "handoff_cancelled_at_gate_1",
                             "reason": "dropped by the founder", "result": "not run"})
    if not workers:
        return {"status": "no_workers_selected", "brief": brief, "handoffs": handoffs,
                "rationale": "every assigned analyst was dropped at Gate 1"}

    # --- parallel fan-out: launch every future first, then resolve --------
    degraded = not CORPUS_OK
    fanout_started = time.time()
    futures = {n: analyst(n, brief, degraded) for n in workers}
    parts, failed = {}, []
    for n, f in futures.items():
        try:
            parts[n] = f.result()
            handoffs.append({"from": n, "to": "supervisor", "via": "transfer_back_to_supervisor",
                             "reason": "analysis complete", "result": "returned WorkerAnalysis"})
        except Exception as e:
            failed.append(n)
            handoffs.append({"from": n, "to": "supervisor", "via": "transfer_back_to_supervisor",
                             "reason": f"failed after retries: {type(e).__name__}",
                             "result": "no analysis"})
            print(f"[graceful partial] {n} failed: {type(e).__name__}: {e}")
    fanout_wall = round(time.time() - fanout_started, 1)
    worker_seconds = round(sum(a.get("_seconds", 0) for a in parts.values()), 1)

    report = synthesize(brief, parts, prior, degraded, failed).result()

    # Deterministic honesty. evidence_mode is COMPUTED from the findings that survived
    # verify_claims(), not asserted by the model -- an earlier run with a healthy corpus still
    # reported "degraded" simply because the model felt unsure. `degraded` here means the CORPUS
    # was unavailable; a competitor analyst can still cite live URLs in that state, which is why
    # T4 checks corpus chunk ids specifically rather than the absence of all citations.
    found = [c for a in parts.values() for c in (a.get("findings") or [])]
    n_grounded = sum(1 for c in found if c["basis"] == "evidence")
    rate = n_grounded / len(found) if found else 0.0
    report["evidence_mode"] = ("degraded" if degraded or rate == 0
                               else "grounded" if rate >= 0.6 else "partial")
    if report["evidence_mode"] == "degraded":
        report["overall_confidence"] = min(_to_float(report["overall_confidence"]), 0.35)

    proposal = memory_proposal(brief, report).result()

    # ===== GATE 2: interrupt() before the irreversible write ==============
    consent = _approval(interrupt({
        "gate": "memory_write",
        "question": "May I remember this for your future sessions?",
        "proposed_writes": proposal, "namespace": f"('founder', '{uid}')",
        "reply_with": "Command(resume={'approved': bool, "
                      "'redact_memory_keys': ['profile'|'idea_entry']})"}))
    written = (persist_memory(uid, proposal, consent.redact_memory_keys).result()
               if consent.approved else {"written": [], "note": "founder declined"})

    return {"status": "complete", "brief": brief, "routing": decision.model_dump(),
            "routed_via": routed["via"], "handoffs": handoffs,
            "workers_run": list(parts), "workers_failed": failed,
            "analyses": parts, "report": report, "memory": written,
            "grounding_rate": round(rate, 3),
            "fanout_wall_seconds": fanout_wall, "worker_seconds_total": worker_seconds}

print("analyze_idea ready - Orchestrator-Worker, handoff routing, two gates")

## 6. Demos - display helpers

In [ ]:
def show_gate(result):
    """Print a pending interrupt."""
    if "__interrupt__" not in result:
        print("no interrupt pending"); return None
    p = result["__interrupt__"][0].value
    print(f"~~~ PAUSED at gate: {p['gate']} ~~~\n{p['question']}\n")
    if p.get("handoffs"):
        print("handoff tool calls the LLM emitted:")
        for h in p["handoffs"]:
            print(f"  {h['via']:<24} {h['from']} -> {h['to']} :: {str(h['reason'])[:70]}")
        print()
    for k, v in p.items():
        if k not in ("gate", "question", "reply_with", "handoffs"):
            print(f"{k}: {json.dumps(v, ensure_ascii=False)[:900]}")
    print("\nreply with:", p["reply_with"])
    return p


def show_handoffs(result):
    """The full handoff ledger for one run -- who the LLM chose, and who reported back."""
    print("HANDOFF LEDGER")
    for h in result.get("handoffs", []):
        print(f"  {h['via']:<28} {h['from']:<11} -> {h['to']:<11} :: {str(h['reason'])[:60]}")


def show_report(result):
    if result.get("status") != "complete":
        print("STATUS:", result["status"])
        print(json.dumps(result, ensure_ascii=False, indent=2)[:1500]); return
    r = result["report"]
    print("=" * 76)
    print(r["idea_one_liner"])
    print("=" * 76)
    print(f"VERDICT    : {r['verdict'].upper()}"
          + (f"  --  {r['verdict_condition']}" if r["verdict_condition"] else ""))
    print(f"CONFIDENCE : {r['overall_confidence']:.2f}   evidence_mode: {r['evidence_mode']}"
          + (f"   grounding: {result['grounding_rate']:.0%}" if "grounding_rate" in result else ""))
    print(f"ANALYSTS   : {result['workers_run']}  (routed via {result.get('routed_via')})"
          + (f"   FAILED: {result['workers_failed']}" if result["workers_failed"] else ""))
    if result.get("fanout_wall_seconds"):
        w, tot = result["fanout_wall_seconds"], result["worker_seconds_total"]
        print(f"PARALLELISM: {tot}s of analyst work finished in {w}s of wall clock"
              f"  ({tot / w:.1f}x concurrent)")
    if r.get("prior_idea_note"):
        print(f"PRIOR IDEAS: {r['prior_idea_note']}")
    for lbl, k in [("MARKET", "market_opportunity"), ("COMPETITORS", "competitors"),
                   ("BUSINESS MODEL", "business_model"),
                   ("SETUP & LICENSING", "setup_and_licensing")]:
        print(f"\n--- {lbl} ---\n{r[k]}")
    for lbl, k in [("STRENGTHS", "strengths"), ("WEAKNESSES", "weaknesses"),
                   ("KEY RISKS", "key_risks"),
                   ("WHAT WOULD CHANGE OUR MIND", "what_would_change_our_mind"),
                   ("NEXT 7 DAYS", "next_7_days")]:
        print(f"\n--- {lbl} ---")
        for x in (r.get(k) or []):
            print("  -", x)
    print("\nMEMORY:", result["memory"])


def all_findings(result):
    """Every Claim from every analyst, flattened -- used by T1."""
    return [c for a in (result.get("analyses") or {}).values()
            for c in (a.get("findings") or [])]


_run_seq = {"n": 0}

def run_full(idea, thread, user, persist=False):
    """Drive one run through both gates.

    The thread id gets a counter suffix, so re-running a cell after a crash always starts a
    fresh checkpoint instead of resuming a half-finished one. Demos that need a STABLE thread
    id (Demo 1, Demo 2, and thread B of T3) call analyze_idea directly, not this.

    Only sends a resume while a gate is actually pending, so an early exit
    (needs_clarification, cancelled_by_founder, no_workers_selected) never gets a
    Command(resume=...) fired into a finished thread.
    """
    _run_seq["n"] += 1
    cfg = {"configurable": {"thread_id": f"{thread}-{_run_seq['n']}"}}
    res = analyze_idea.invoke({"idea": idea, "user_id": user}, cfg)
    for value in ({"approved": True}, {"approved": persist}):
        if not isinstance(res, dict) or "__interrupt__" not in res:
            break
        res = analyze_idea.invoke(Command(resume=value), cfg)
    return res

print("helpers ready")

### Demo 1 - both gates

Rubric Section 5. It pauses, we resume, it pauses again, we resume again. Both `interrupt()` and
`Command(resume=...)` are executed, with output captured below each cell.

In [ ]:
IDEA = ("A subscription service delivering healthy halal meal-prep boxes to busy office "
        "professionals in Riyadh. I want a full validation of the whole idea.")
cfg1 = {"configurable": {"thread_id": "demo-1"}}

r = analyze_idea.invoke({"idea": IDEA, "user_id": "mayar"}, cfg1)
_ = show_gate(r)

In [ ]:
# Gate 1 -> resume: approve the brief as read.
r = analyze_idea.invoke(Command(resume={"approved": True}), cfg1)
_ = show_gate(r)

In [ ]:
# Gate 2 -> resume: both interrupts have now fired AND both have been resumed.
result1 = analyze_idea.invoke(Command(resume={"approved": True}), cfg1)
show_report(result1)
print()
show_handoffs(result1)

### Demo 2 - the resume must change the outcome

A resume that changes nothing proves nothing. Here the founder corrects the business model and
drops an analyst at Gate 1, so a different report comes out: the brief is re-derived from the
correction and the dropped worker never runs.

In [ ]:
cfg2 = {"configurable": {"thread_id": "demo-2"}}
r = analyze_idea.invoke({"idea": IDEA, "user_id": "mayar"}, cfg2)
p = r["__interrupt__"][0].value
print("supervisor's original plan:", p["workers"], "| model:", p["brief"]["business_model_guess"])

analyze_idea.invoke(Command(resume={
    "approved": True,
    "corrections": "This is B2B corporate catering sold to company HR departments on annual "
                   "contracts, NOT a B2C consumer subscription.",
    "drop_workers": ["competitor"]}), cfg2)

result2 = analyze_idea.invoke(Command(resume={"approved": True}), cfg2)

print("\n### what the correction changed ###")
for tag, res in [("run 1", result1), ("run 2", result2)]:
    print(f"{tag}: model={res['brief']['business_model_guess']:<12} "
          f"analysts={res['workers_run']}  verdict={res['report']['verdict']}")

### Demo 3 - routing is a real decision

Rubric Section 2. Three ideas, three routes, decided by the LLM's own `transfer_to_*` tool calls from
the structured brief. The vague one calls `request_clarification` instead, and no worker runs
at all.

In [ ]:
CASES = [
    ("narrow -> competitors only",
     "I already know there's demand for a padel court booking app in Jeddah. I only want to "
     "know who else is doing this."),
    ("narrow -> business only",
     "I run a small home bakery in Dammam with paying customers. I just need to know what "
     "licences I need and how to price for profit."),
    ("too vague -> clarify",
     "I want to build an app. Something with AI. Maybe for businesses."),
]

for i, (label, idea) in enumerate(CASES):
    r = analyze_idea.invoke({"idea": idea, "user_id": "router-test"},
                            {"configurable": {"thread_id": f"route-{i}"}})
    p = r["__interrupt__"][0].value

    print(f"--- {label}")
    print(f"    gate       : {p['gate']}")
    print(f"    analyzable : {p['brief']['is_analyzable']}")
    print("    handoffs   :", [h["via"] for h in p.get("handoffs", [])] or "(structured fallback)")
    # A vague idea stops at the clarification gate, whose payload carries questions but no route
    # or workers -- because no routing decision has been committed to yet.
    if p["gate"] == "clarification":
        print("    route      : clarify - no analyst assigned; the supervisor asks first")
        for q in p["questions"]:
            print("       ?", q)
    else:
        print(f"    route      : {p['route']}   workers: {p['workers']}")
    print(f"    why        : {p['rationale'][:150]}\n")

## 7. Tests

### T1 - Grounding

Counts `Claim`s that survived `verify_claims()` with a real source. Because unverifiable
citations were already downgraded, this cannot be inflated by a model inventing chunk ids.

In [ ]:
# Reuse Demo 1's run rather than burning another full pipeline -- Groq's free tier is
# 200k tokens/day and this notebook runs the orchestrator several times.
res = result1
claims = all_findings(res)
grounded = [c for c in claims if c["basis"] == "evidence"]
rate = len(grounded) / len(claims) if claims else 0

print(f"claims: {len(claims)}   grounded: {len(grounded)}   rate: {rate:.0%}")
print(f"evidence_mode: {res['report']['evidence_mode']}\n")
for c in claims[:6]:
    tag = "EVIDENCE  " if c["basis"] == "evidence" else "assumption"
    print(f"[{tag}] {c['source_id'] or '-'}\n             {c['statement'][:110]}")
print(f"\nT1 {'PASS' if rate >= 0.6 else 'REVIEW'} - target is 60% grounded")

### T2 - Anti-flattery: can it say no?

The test that matters most. A saturated market, no differentiation, no wedge. If this returns
`go`, the system is a flattery machine and the whole premise fails.

In [ ]:
BAD = ("A ride-hailing app for Riyadh, exactly like Uber and Careem but with a nicer logo. "
       "No price advantage, no driver supply advantage, no niche. Full validation please.")

rep = run_full(BAD, "t2", "t2-user")["report"]
print("verdict   :", rep["verdict"])
print("condition :", rep["verdict_condition"] or "(none)")
print("confidence:", rep["overall_confidence"])
print("\nweaknesses:")
for w in rep["weaknesses"]:
    print("  -", w)
print("\nwhat would change our mind:")
for w in rep["what_would_change_our_mind"]:
    print("  -", w)
print(f"\nT2 {'PASS' if rep['verdict'] in ('no_go', 'pivot') else 'FAIL'} - expected no_go or pivot")

### T3 - Cross-thread long-term memory

Rubric Section 4. Thread A writes founder facts. Thread B is a **different `thread_id`** - a fresh
checkpointer state with no shared short-term context - and reads them. If it only worked
inside one thread it would be short-term state.

In [ ]:
USER = "t3-founder"
resA = run_full("A same-day grocery delivery startup for Riyadh neighbourhoods. Full validation.",
                "t3-A", USER, persist=True)
print("thread A wrote:", resA["memory"])
print("store now:", json.dumps(load_history(store, USER), ensure_ascii=False))

cfgB = {"configurable": {"thread_id": "t3-B"}}     # different thread, same founder
rB = analyze_idea.invoke({"idea": "Now a same-day pharmacy delivery service for Riyadh. "
                                  "Full validation.", "user_id": USER}, cfgB)
p = rB["__interrupt__"][0].value
print("\nthread B sees what thread A wrote:")
print("  profile    :", json.dumps(p["known_profile"], ensure_ascii=False))
print("  prior ideas:", json.dumps(p["known_prior_ideas"], ensure_ascii=False))

analyze_idea.invoke(Command(resume={"approved": True}), cfgB)
resB = analyze_idea.invoke(Command(resume={"approved": False}), cfgB)

print("\nprior_idea_note in thread B's report:")
print("  ", resB["report"]["prior_idea_note"] or "(empty)")
ok = bool(p["known_profile"]) and len(p["known_prior_ideas"]) >= 1
print(f"\nT3 {'PASS' if ok else 'FAIL'} - thread B read {len(p['known_prior_ideas'])} prior idea(s)")

### T4 - Graceful degradation

Rubric Section 6. Empty the corpus and re-run. The system must complete and confess, not complete and
invent. `degraded` means the **corpus** was unavailable, so the check is that no *corpus chunk
id* appears as evidence - the competitor analyst may still legitimately cite live URLs.

In [ ]:
_saved, _ok = chunks, CORPUS_OK
chunks, CORPUS_OK = [], False
print("corpus emptied.")
res = run_full(IDEA, "t4", "t4-user")
rep = res["report"]

cited = [c["source_id"] for c in all_findings(res)
         if c["basis"] == "evidence" and c["source_id"] in CHUNK_IDS]
ok = (res["status"] == "complete" and rep["evidence_mode"] == "degraded"
      and rep["overall_confidence"] <= 0.35 and not cited)

print("\nstatus            :", res["status"])
print("evidence_mode     :", rep["evidence_mode"])
print("overall_confidence:", rep["overall_confidence"])
print("corpus citations  :", cited, "(should be empty)")
print("\nwhat it tells the founder:\n  ", rep["market_opportunity"][:350])
print(f"\nT4 {'PASS' if ok else 'FAIL'} - degraded without fabricating grounding")

chunks, CORPUS_OK = _saved, _ok
if chunks:
    vector_store = InMemoryVectorStore(embedding=embeddings)
    vector_store.add_documents(chunks)
print(f"\ncorpus restored: {len(chunks)} chunks, CORPUS_OK = {CORPUS_OK}")

## 8. LangSmith - what tracing showed us

Rubric Section 8. Tracing is enabled in Section 0.1 through `LANGCHAIN_TRACING_V2` (not
`LANGSMITH_TRACING_V2`, which produces no trace and no error), and we probe the key once so a
rejected key fails loudly instead of 403-ing on every run.

Every claim below is backed either by a cell output in this file or by a span in the trace for
this run; we say which.

1. **A silent double call on every analyst.** *(trace, now also visible in-file)* The trace
   showed each `WorkerAnalysis` call appearing twice - a `function_calling` attempt that 400'd,
   then a `json_mode` attempt that succeeded. The notebook printed only a bare
   `BadRequestError` class name, so the run *looked* healthy. The cause was `anyOf` in the JSON
   schema, emitted by the `Union[float, str]` and `Optional[List[...]]` fields we had widened
   to tolerate sloppy models. Narrowing the schema removed the wasted call, and Section 1 now prints
   `anyOf=0` for all six contracts so the fix is visible without opening a trace.
   `structured()` also prints the API's own message now, so the next failure of this kind is
   legible on the page.
2. **Retries are real, and countable.** *(in-file: Section 4.4)* `flaky_probe` prints `attempt 1`,
   `attempt 2`, `attempt 3` and then succeeds, with no loop and no `sleep` in the body - which
   is what distinguishes a `RetryPolicy` object from a hand-written retry. In the trace the
   same three attempts appear nested under one task span.
3. **The fan-out is genuinely concurrent.** *(in-file: the `PARALLELISM` line in every report)*
   We stopped asserting this and started measuring it: each analyst records its own duration
   and the orchestrator records the wall clock around the fan-out. The report prints both, so
   the concurrency factor is a number on the page rather than a claim. The trace corroborates
   it - the analyst spans overlap in the timeline instead of running end to end.
4. **Evidence gathering is the fragile half, and it fails loudly now.** *(in-file: the
   `[run_tools]` lines in the analyst output)* Under a three-worker fan-out the small model
   hits 429s, which the trace showed as failed spans that ended a round early. The loop used to
   `break` there and hand the analyst an empty evidence block - an entire analysis silently
   reduced to assumptions. It now falls through to an unforced call and then to the main model,
   and prints each fallback, so a degraded round is visible rather than inferred.

## 9. What we would do differently

- **Trust the deterministic layer earlier.** Two of the bugs that cost us a full run were
  invisible because a guard silently swallowed them: a mis-escaped regex that made every
  citation unverifiable, and a namespace mismatch that made every corpus search return
  `NO_RESULTS`. Both failed *quietly and safely*, which is the worst combination - the run
  completed, the report read well, and the grounding rate was 0%. The smoke tests at the end of
  Section 2.2 and Section 3 exist so neither failure can ever be silent again.
- **Do not widen a schema to survive a bad model.** Widening pushed the problem to the API
  boundary, where it became a hard 400 on every analyst call. Narrow types plus a coercing
  `mode="before"` validator gave us the same tolerance with none of the rejection.
- **Prompts should be clear, not defensive.** Our earlier analyst prompt spent six lines
  threatening the model about invented source ids. It did not improve citation accuracy -
  `verify_claims()` did, because it is code, not persuasion. The prompt got shorter and the
  grounding rate went up.
- **Measure the claim instead of making it.** The concurrency line in Section 8 exists because we
  could not otherwise show a reader, inside this file, that the fan-out was parallel.
- **Next:** persist the Store to disk so memory survives a runtime restart, and add an
  Evaluator-Optimizer loop that re-runs a worker when its grounding rate falls below a
  threshold.

In [ ]:
print("LangSmith tracing configuration (evidence for the Section 8 write-up)")
print("  LANGCHAIN_TRACING_V2 :", os.environ.get("LANGCHAIN_TRACING_V2"))
print("  LANGCHAIN_PROJECT    :", os.environ.get("LANGCHAIN_PROJECT"))
print("  LANGCHAIN_ENDPOINT   :", os.environ.get("LANGCHAIN_ENDPOINT", "(default US)"))
print("  key present          :", bool(os.environ.get("LANGCHAIN_API_KEY")))
print("\nThe four error strategies, and where each one lives:")
for n, (name, where) in enumerate([
        ("RetryPolicy on every LLM task",        "Section 4 RETRY, Section 4.4 flaky_probe"),
        ("Fallback to degraded mode",            "Section 4.3 synthesize + Section 5 evidence_mode, T4"),
        ("Graceful partial (worker unavailable)","Section 5 fan-out except branch"),
        ("Retry on a different model / method",  "Section 1.1 structured()")], 1):
    print(f"  {n}. {name:<40} {where}")

## 10. Preflight self-check

The submission checklist, machine-checked where a machine can check it. This is the cell that
makes our headline rule enforceable: **every claim in the write-up is backed by something
visible in the file.** The unchecked items at the bottom are the manual steps that live outside
this notebook.

In [ ]:
PROJECT, NAME = "IdeaFoundry", "Mayar Alhindi"
TRACK, PATTERN = "A - Supervisor + Workers", "Orchestrator-Worker"

_no_anyof = all(json.dumps(m.model_json_schema()).count("anyOf") == 0 for m in
                (Claim, IdeaBrief, RoutingDecision, WorkerAnalysis, ApprovalDecision, FinalReport))

_checks = [
    ("Project, name, track and pattern declared",  all(map(bool, (PROJECT, NAME, TRACK, PATTERN)))),
    ("Cohort dates filled in (not a placeholder)", "<<FILL IN" not in COHORT),
    ("Keys resolved from Colab Secrets, never a cell",
                                                   bool(groq_src) and "typed" not in str(groq_src)),
    ("Tracing variable is LANGCHAIN_TRACING_V2",   "LANGCHAIN_TRACING_V2" in os.environ),
    ("Tracing actually enabled",                   os.environ.get("LANGCHAIN_TRACING_V2") == "true"),
    ("RAG: retriever returns results",             bool(retrieve("commercial registration", k=1))),
    ("RAG: citation parser extracts source ids",
                                        offered_ids("[source_id: x::0001]\ny") == {"x::0001"}),
    ("RAG: corpus covers both namespaces",         len(counters) >= 2),
    ("Schemas emit no anyOf at the API boundary",  _no_anyof),
    ("Built with @task / @entrypoint",             hasattr(analyze_idea, "invoke")),
    ("interrupt() AND Command(resume) executed",   result1.get("status") == "complete"),
    ("Routing via LLM handoff tool calls",         result1.get("routed_via") == "tool_calls"),
    ("Handoff ledger recorded, both directions",
                    any(h["via"].startswith("transfer_to_") for h in result1.get("handoffs", []))
                and any(h["via"] == "transfer_back_to_supervisor"
                        for h in result1.get("handoffs", []))),
    ("Fan-out measurably concurrent",
                    result1.get("worker_seconds_total", 0) > result1.get("fanout_wall_seconds", 1)),
    ("Grounding at or above 60% (T1)",             result1.get("grounding_rate", 0) >= 0.6),
    ("Cross-thread long-term memory (T3)",         len(load_history(store, "t3-founder")) >= 1),
]
_passed = sum(1 for _, ok in _checks if ok)
print(f"{PROJECT} - preflight   |   Track {TRACK}   |   pattern: {PATTERN}\n")
for label, ok in _checks:
    print(f"  [{'x' if ok else ' '}] {label}")
print(f"\n{_passed}/{len(_checks)} automated checks passed")

print("\nManual steps, in the repository rather than this notebook:")
for item in ["Restart the kernel and Run all, top to bottom, so every cell carries its output",
             "README.md describing the project and how to run it",
             ".gitignore excluding secrets and generated files",
             "Confirm no API key appears anywhere in git history"]:
    print("  [ ]", item)